# NLP Course Project: Transformer — From Scratch vs. Pretrained Model

**Task:** 3-Class Sentiment Analysis (`Negative`, `Neutral`, `Positive`)

### Project Goal
Build and fairly compare two Transformer classifiers on the **exact same data splits** (`seed = 42`):
1. **Model A — Scratch Transformer (Encoder-Only):** Built from basic PyTorch primitives (`nn.Embedding`, `nn.Linear`, `nn.LayerNorm`, `nn.Dropout`, `torch.matmul`, `torch.softmax`) with **zero** pretrained weights and **without** `nn.Transformer`, `nn.TransformerEncoder`, or `nn.MultiheadAttention`.
2. **Model B — Pretrained Transformer (`distilbert-base-uncased`):** Fine-tuned cleanly on the exact same training/validation splits.

### Dataset Citation
- **Dataset:** `cardiffnlp/tweet_eval` (`sentiment` configuration — SemEval-2017 Task 4A)
- **Direct URL:** https://huggingface.co/datasets/cardiffnlp/tweet_eval
- **License:** MIT License (TweetEval benchmark) / SemEval-2017 Task 4A research data
- **Reference:** Barbieri et al., *TweetEval: Unified Benchmark and Comparative Evaluation for Tweet Classification* (Findings of EMNLP 2020).

### Artifacts Generated by This Notebook
- `scratch_model.pt` — Trained Model A weights + architecture config
- `vocab.json` — Word-level vocabulary built strictly on the training split
- `pretrained_model/` — Fine-tuned DistilBERT weights and tokenizer
- `results.json` — Held-out test metrics, latency, parameter counts, training times, and confusion matrices (loaded by `app.py`)

In [ ]:
import copy
import json
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# --- Global Configuration ---
SEED = 42
LABELS = ["Negative", "Neutral", "Positive"]
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}
LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}

DATASET_NAME = "cardiffnlp/tweet_eval"
DATASET_CONFIG = "sentiment"
DATASET_URL = "https://huggingface.co/datasets/cardiffnlp/tweet_eval"
DATASET_LICENSE = "MIT (TweetEval benchmark) / SemEval-2017 Task 4A"

MAX_LEN = 64
PRETRAINED_CHECKPOINT = "distilbert-base-uncased"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | PyTorch: {torch.__version__}")

## 1. Dataset Loading & Balanced Train / Validation / Test Splits

We load the official `train`, `validation`, and `test` splits from `cardiffnlp/tweet_eval` (`sentiment`) and sample a **strictly balanced** subset from each split (using `SEED = 42`):
- **Train:** 6,000 samples (2,000 per class)
- **Validation:** 900 samples (300 per class)
- **Test:** 1,500 samples (500 per class)

Because the splits come from disjoint official partitions, there is **zero data leakage**. The test set is never touched until final evaluation.

In [ ]:
raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(f"Dataset : {DATASET_NAME} ({DATASET_CONFIG})")
print(f"URL     : {DATASET_URL}")
print(f"License : {DATASET_LICENSE}")
print("Official split sizes:", {k: len(v) for k, v in raw_dataset.items()})


def sample_balanced_split(hf_split, per_class: int, seed: int = 42):
    """Sample an equal number of examples per class with a fixed random seed."""
    rng = random.Random(seed)
    by_class = {0: [], 1: [], 2: []}
    for idx, label in enumerate(hf_split["label"]):
        by_class[int(label)].append(idx)

    selected_indices = []
    for cls_id in [0, 1, 2]:
        indices = by_class[cls_id]
        rng.shuffle(indices)
        take = min(per_class, len(indices))
        selected_indices.extend(indices[:take])

    rng.shuffle(selected_indices)
    texts = [hf_split[i]["text"].strip() for i in selected_indices]
    labels = [int(hf_split[i]["label"]) for i in selected_indices]
    return texts, labels


train_texts, train_labels = sample_balanced_split(raw_dataset["train"], per_class=2000, seed=SEED)
val_texts, val_labels = sample_balanced_split(raw_dataset["validation"], per_class=300, seed=SEED)
test_texts, test_labels = sample_balanced_split(raw_dataset["test"], per_class=500, seed=SEED)


def get_class_distribution(labels):
    counts = Counter(labels)
    return {LABELS[i]: counts.get(i, 0) for i in range(len(LABELS))}


class_dist = {
    "train": get_class_distribution(train_labels),
    "validation": get_class_distribution(val_labels),
    "test": get_class_distribution(test_labels),
}

print("\nBalanced Subset Class Distribution:")
display(pd.DataFrame(class_dist))

print("\nSample Tweets from Training Set:")
for i in range(5):
    print(f"  [{LABELS[train_labels[i]]:8s}] {train_texts[i]}")

## 2. Model A — Encoder-Only Transformer From Scratch (Pure PyTorch)

We build every Transformer component explicitly using basic PyTorch building blocks (`nn.Embedding`, `nn.Linear`, `nn.LayerNorm`, `nn.Dropout`, `torch.matmul`, `torch.softmax`):
1. **Token Embedding + Sinusoidal Positional Encoding**
2. **Scaled Dot-Product Attention** with **Padding Mask**
3. **Multi-Head Self-Attention** (`n_heads = 4`)
4. **Position-Wise Feed-Forward Network** (`d_model = 128`, `d_ff = 256`)
5. **Residual Connections + Layer Normalization** (`n_layers = 2`)
6. **Masked Mean Pooling + 3-Class Linear Head**

> **Strict Compliance:** No `nn.Transformer`, `nn.TransformerEncoder`, `nn.TransformerEncoderLayer`, `nn.MultiheadAttention`, or pretrained weights are used anywhere in Model A.

In [ ]:
# --- Word-Level Tokenizer & Vocabulary (Built Strictly on Train Split) ---
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MAX_VOCAB_SIZE = 10000
MIN_FREQ = 2


def tokenize(text: str):
    """Lowercase regex tokenizer preserving sentiment punctuation (!, ?) and contractions."""
    return re.findall(r"[a-z0-9']+|[!?.,]", text.lower())


def build_vocab(texts, max_vocab_size: int = 10000, min_freq: int = 2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for word, freq in counter.most_common(max_vocab_size - 2):
        if freq < min_freq:
            break
        vocab[word] = len(vocab)
    return vocab


def encode_text(text: str, vocab: dict, max_len: int = MAX_LEN):
    """Map text to fixed-length token IDs (guarantees >= 1 non-pad token)."""
    tokens = tokenize(text)
    unk_id = vocab[UNK_TOKEN]
    pad_id = vocab[PAD_TOKEN]
    ids = [vocab.get(tok, unk_id) for tok in tokens[:max_len]]
    if not ids:
        ids = [unk_id]
    if len(ids) < max_len:
        ids = ids + [pad_id] * (max_len - len(ids))
    return ids


vocab = build_vocab(train_texts, max_vocab_size=MAX_VOCAB_SIZE, min_freq=MIN_FREQ)
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=2)
print(f"Vocabulary size: {len(vocab):,} (saved to vocab.json)")


class ScratchTextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.encoded = [encode_text(t, vocab, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encoded[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


SCRATCH_BATCH_SIZE = 64
scratch_train_loader = DataLoader(
    ScratchTextDataset(train_texts, train_labels, vocab),
    batch_size=SCRATCH_BATCH_SIZE,
    shuffle=True,
)
scratch_val_loader = DataLoader(
    ScratchTextDataset(val_texts, val_labels, vocab),
    batch_size=SCRATCH_BATCH_SIZE,
    shuffle=False,
)
scratch_test_loader = DataLoader(
    ScratchTextDataset(test_texts, test_labels, vocab),
    batch_size=SCRATCH_BATCH_SIZE,
    shuffle=False,
)

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    """Compute Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V with optional padding mask."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        # mask == 0 marks padding positions so softmax assigns them ~0 weight
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = torch.softmax(scores, dim=-1)
    if dropout is not None:
        attn_weights = dropout(attn_weights)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights


class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self-Attention implemented from scratch with nn.Linear projections."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()

        # Project and split into (B, n_heads, L, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask=mask, dropout=self.dropout)

        # Concatenate heads back to (B, L, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_out)


class PositionWiseFFN(nn.Module):
    """Two-layer position-wise Feed-Forward Network."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.dropout(self.act(self.fc1(x))))


class TransformerEncoderBlock(nn.Module):
    """Single Encoder layer: Self-Attention + Residual/LayerNorm + FFN + Residual/LayerNorm."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)

        self.ffn = PositionWiseFFN(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        x = self.norm1(x + self.drop1(self.self_attn(x, mask=mask)))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x


class ScratchTransformerClassifier(nn.Module):
    """Complete Encoder-only Transformer for 3-class Sentiment Classification."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        d_ff: int = 256,
        n_layers: int = 2,
        max_len: int = 64,
        num_classes: int = 3,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # Padding mask: (B, 1, 1, L) where 1 = real token, 0 = <PAD>
        pad_mask = (input_ids != self.pad_idx).unsqueeze(1).unsqueeze(2)

        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.embed_dropout(self.pos_encoding(x))

        for layer in self.layers:
            x = layer(x, mask=pad_mask)

        # Masked Mean Pooling across valid (non-pad) tokens
        valid_mask = (input_ids != self.pad_idx).unsqueeze(-1).float()  # (B, L, 1)
        pooled = (x * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1.0)
        return self.classifier(pooled)


# Instantiate Model A
scratch_config = {
    "vocab_size": len(vocab),
    "d_model": 128,
    "n_heads": 4,
    "d_ff": 256,
    "n_layers": 2,
    "max_len": MAX_LEN,
    "num_classes": len(LABELS),
    "dropout": 0.2,
    "pad_idx": vocab[PAD_TOKEN],
}

set_seed(SEED)
scratch_model = ScratchTransformerClassifier(**scratch_config).to(device)

scratch_total_params = sum(p.numel() for p in scratch_model.parameters())
scratch_trainable_params = sum(p.numel() for p in scratch_model.parameters() if p.requires_grad)
print(f"Model A (Scratch Transformer) — Total Params: {scratch_total_params:,} | Trainable: {scratch_trainable_params:,}")

# Quick forward-pass shape check
sample_batch, _ = next(iter(scratch_train_loader))
with torch.no_grad():
    sample_logits = scratch_model(sample_batch.to(device))
print(f"Sanity check — Input shape: {tuple(sample_batch.shape)} -> Logits shape: {tuple(sample_logits.shape)}")

In [ ]:
# --- Train Model A (Scratch Transformer) ---
SCRATCH_EPOCHS = 10
SCRATCH_LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer_a = torch.optim.AdamW(scratch_model.parameters(), lr=SCRATCH_LR, weight_decay=1e-2)


def eval_scratch_loader(model, loader):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_targets.extend(labels.cpu().tolist())
    avg_loss = total_loss / len(all_targets)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, macro_f1, all_targets, all_preds


scratch_history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1_a = -1.0
best_state_a = None

start_time_a = time.perf_counter()
for epoch in range(1, SCRATCH_EPOCHS + 1):
    scratch_model.train()
    running_loss = 0.0
    for input_ids, labels in scratch_train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        optimizer_a.zero_grad()
        logits = scratch_model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(scratch_model.parameters(), max_norm=1.0)
        optimizer_a.step()
        running_loss += loss.item() * labels.size(0)

    train_loss = running_loss / len(train_labels)
    val_loss, val_f1, _, _ = eval_scratch_loader(scratch_model, scratch_val_loader)

    scratch_history["train_loss"].append(round(train_loss, 4))
    scratch_history["val_loss"].append(round(val_loss, 4))
    scratch_history["val_f1"].append(round(val_f1, 4))

    if val_f1 > best_val_f1_a:
        best_val_f1_a = val_f1
        best_state_a = copy.deepcopy(scratch_model.state_dict())

    print(f"Epoch {epoch:2d}/{SCRATCH_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

scratch_train_time = time.perf_counter() - start_time_a

# Restore best validation checkpoint and save to scratch_model.pt
scratch_model.load_state_dict(best_state_a)
torch.save({"model_state_dict": scratch_model.state_dict(), "config": scratch_config}, "scratch_model.pt")
print(f"\nModel A training complete in {scratch_train_time:.1f}s | Best Val Macro-F1: {best_val_f1_a:.4f}")
print("Saved Model A checkpoint to scratch_model.pt")

## 3. Model B — Pretrained Transformer (`distilbert-base-uncased`)

Now we fine-tune **`distilbert-base-uncased`** (6 Transformer layers, `d_model = 768`, 12 attention heads, ~66.9M parameters) using its official WordPiece tokenizer on the **exact same 6,000 train / 900 validation tweets**:
- **Epochs:** 3
- **Batch Size:** 32
- **Optimizer:** `AdamW(lr=2e-5, weight_decay=0.01)` with 10% linear warmup
- **Checkpoint Selection:** Best validation Macro-F1 saved to `pretrained_model/`

In [ ]:
pretrained_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_CHECKPOINT)


class PretrainedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }


PRETRAINED_BATCH_SIZE = 32
pretrained_train_loader = DataLoader(
    PretrainedTextDataset(train_texts, train_labels, pretrained_tokenizer),
    batch_size=PRETRAINED_BATCH_SIZE,
    shuffle=True,
)
pretrained_val_loader = DataLoader(
    PretrainedTextDataset(val_texts, val_labels, pretrained_tokenizer),
    batch_size=PRETRAINED_BATCH_SIZE,
    shuffle=False,
)
pretrained_test_loader = DataLoader(
    PretrainedTextDataset(test_texts, test_labels, pretrained_tokenizer),
    batch_size=PRETRAINED_BATCH_SIZE,
    shuffle=False,
)

set_seed(SEED)
pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED_CHECKPOINT,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

pretrained_total_params = sum(p.numel() for p in pretrained_model.parameters())
pretrained_trainable_params = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)
print(
    f"Model B ({PRETRAINED_CHECKPOINT}) — Total Params: {pretrained_total_params:,} | "
    f"Trainable: {pretrained_trainable_params:,}"
)

In [ ]:
# --- Fine-Tune Model B (Pretrained DistilBERT) ---
PRETRAINED_EPOCHS = 3
PRETRAINED_LR = 2e-5

optimizer_b = torch.optim.AdamW(pretrained_model.parameters(), lr=PRETRAINED_LR, weight_decay=0.01)
total_steps = len(pretrained_train_loader) * PRETRAINED_EPOCHS
scheduler_b = get_linear_schedule_with_warmup(
    optimizer_b,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


def eval_pretrained_loader(model, loader):
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item() * labels.size(0)
            all_preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
            all_targets.extend(labels.cpu().tolist())
    avg_loss = total_loss / len(all_targets)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, macro_f1, all_targets, all_preds


pretrained_history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1_b = -1.0
best_state_b = None

start_time_b = time.perf_counter()
for epoch in range(1, PRETRAINED_EPOCHS + 1):
    pretrained_model.train()
    running_loss = 0.0
    for batch in pretrained_train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_b.zero_grad()
        outputs = pretrained_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(pretrained_model.parameters(), max_norm=1.0)
        optimizer_b.step()
        scheduler_b.step()
        running_loss += loss.item() * labels.size(0)

    train_loss = running_loss / len(train_labels)
    val_loss, val_f1, _, _ = eval_pretrained_loader(pretrained_model, pretrained_val_loader)

    pretrained_history["train_loss"].append(round(train_loss, 4))
    pretrained_history["val_loss"].append(round(val_loss, 4))
    pretrained_history["val_f1"].append(round(val_f1, 4))

    if val_f1 > best_val_f1_b:
        best_val_f1_b = val_f1
        best_state_b = copy.deepcopy(pretrained_model.state_dict())

    print(f"Epoch {epoch:2d}/{PRETRAINED_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

pretrained_train_time = time.perf_counter() - start_time_b

# Restore best validation checkpoint and save to pretrained_model/
pretrained_model.load_state_dict(best_state_b)
Path("pretrained_model").mkdir(exist_ok=True)
pretrained_model.save_pretrained("pretrained_model")
pretrained_tokenizer.save_pretrained("pretrained_model")
print(f"\nModel B fine-tuning complete in {pretrained_train_time:.1f}s | Best Val Macro-F1: {best_val_f1_b:.4f}")
print("Saved Model B checkpoint & tokenizer to pretrained_model/")

## 4. Final Evaluation on the Identical Held-Out Test Split (1,500 Samples)

We compute all required classification and efficiency metrics on the exact same held-out test set:
- **Classification:** Accuracy, Macro-F1, Macro Precision, Macro Recall, Confusion Matrix
- **Efficiency:** Average single-sample inference latency (`ms`), Total & Trainable parameter counts, Training wall-clock time (`s`)

All measured numbers are saved to `results.json` for `app.py`.

In [ ]:
def compute_classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist(),
    }


def measure_scratch_latency(model, texts, vocab, n_samples=200):
    """Measure average single-sample inference latency in milliseconds."""
    model.eval()
    subset = texts[:n_samples]

    # Warmup passes
    for t in subset[:10]:
        x = torch.tensor([encode_text(t, vocab, MAX_LEN)], dtype=torch.long, device=device)
        with torch.no_grad():
            _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for t in subset:
            x = torch.tensor([encode_text(t, vocab, MAX_LEN)], dtype=torch.long, device=device)
            _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return ((time.perf_counter() - start) / len(subset)) * 1000.0


def measure_pretrained_latency(model, tokenizer, texts, n_samples=200):
    """Measure average single-sample inference latency in milliseconds."""
    model.eval()
    subset = texts[:n_samples]

    # Warmup passes
    for t in subset[:10]:
        enc = tokenizer(t, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            _ = model(**enc)
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for t in subset:
            enc = tokenizer(t, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
            enc = {k: v.to(device) for k, v in enc.items()}
            _ = model(**enc)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return ((time.perf_counter() - start) / len(subset)) * 1000.0


# Run held-out test set evaluation
_, _, y_true_a, y_pred_a = eval_scratch_loader(scratch_model, scratch_test_loader)
_, _, y_true_b, y_pred_b = eval_pretrained_loader(pretrained_model, pretrained_test_loader)
assert y_true_a == y_true_b, "Test targets must match identically!"

metrics_a = compute_classification_metrics(y_true_a, y_pred_a)
metrics_b = compute_classification_metrics(y_true_b, y_pred_b)

latency_a_ms = measure_scratch_latency(scratch_model, test_texts, vocab, n_samples=200)
latency_b_ms = measure_pretrained_latency(pretrained_model, pretrained_tokenizer, test_texts, n_samples=200)

# Save complete results.json
results = {
    "dataset": {
        "name": DATASET_NAME,
        "config": DATASET_CONFIG,
        "url": DATASET_URL,
        "license": DATASET_LICENSE,
        "seed": SEED,
        "labels": LABELS,
        "splits": {
            "train": len(train_labels),
            "validation": len(val_labels),
            "test": len(test_labels),
        },
        "class_distribution": class_dist,
    },
    "device": str(device),
    "models": {
        "scratch": {
            "name": "Model A: Scratch Transformer (Encoder-Only)",
            "architecture": "2-layer Pure PyTorch Encoder (d_model=128, 4 heads, d_ff=256)",
            **metrics_a,
            "avg_latency_ms": round(latency_a_ms, 3),
            "total_params": scratch_total_params,
            "trainable_params": scratch_trainable_params,
            "training_time_sec": round(scratch_train_time, 2),
            "history": scratch_history,
        },
        "pretrained": {
            "name": f"Model B: Pretrained DistilBERT ({PRETRAINED_CHECKPOINT})",
            "checkpoint": PRETRAINED_CHECKPOINT,
            **metrics_b,
            "avg_latency_ms": round(latency_b_ms, 3),
            "total_params": pretrained_total_params,
            "trainable_params": pretrained_trainable_params,
            "training_time_sec": round(pretrained_train_time, 2),
            "history": pretrained_history,
        },
    },
}

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved evaluation metrics to results.json\n")

# Print Classification Reports
print("=== Model A: Scratch Transformer — Classification Report ===")
print(classification_report(y_true_a, y_pred_a, target_names=LABELS, digits=4))

print("=== Model B: Pretrained DistilBERT — Classification Report ===")
print(classification_report(y_true_b, y_pred_b, target_names=LABELS, digits=4))

# Side-by-Side Comparison Table
s_res = results["models"]["scratch"]
p_res = results["models"]["pretrained"]
comparison_df = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Macro-F1",
            "Macro Precision",
            "Macro Recall",
            "Avg Latency (ms/sample)",
            "Total Parameters",
            "Training Time (sec)",
        ],
        "Model A (Scratch)": [
            f"{s_res['accuracy']:.4f}",
            f"{s_res['macro_f1']:.4f}",
            f"{s_res['precision']:.4f}",
            f"{s_res['recall']:.4f}",
            f"{s_res['avg_latency_ms']:.2f}",
            f"{s_res['total_params']:,}",
            f"{s_res['training_time_sec']:.1f}",
        ],
        "Model B (Pretrained DistilBERT)": [
            f"{p_res['accuracy']:.4f}",
            f"{p_res['macro_f1']:.4f}",
            f"{p_res['precision']:.4f}",
            f"{p_res['recall']:.4f}",
            f"{p_res['avg_latency_ms']:.2f}",
            f"{p_res['total_params']:,}",
            f"{p_res['training_time_sec']:.1f}",
        ],
    }
)
print("=== Side-by-Side Test Comparison ===")
display(comparison_df)

In [ ]:
# --- Visualization 1: Classification Metrics Bar Chart ---
metric_names = ["Accuracy", "Macro-F1", "Precision", "Recall"]
vals_a = [s_res["accuracy"], s_res["macro_f1"], s_res["precision"], s_res["recall"]]
vals_b = [p_res["accuracy"], p_res["macro_f1"], p_res["precision"], p_res["recall"]]

x = np.arange(len(metric_names))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - width / 2, vals_a, width, label="Model A: Scratch Transformer", color="#4C78A8")
b2 = ax.bar(x + width / 2, vals_b, width, label="Model B: Pretrained DistilBERT", color="#F58518")
ax.set_ylabel("Score")
ax.set_title("Held-Out Test Set: Scratch Transformer vs. Pretrained DistilBERT")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.08)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(loc="lower right")

for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.3f}", (bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# --- Visualization 2: Side-by-Side Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
configs = [
    (np.array(s_res["confusion_matrix"]), "Model A: Scratch Transformer", "Blues"),
    (np.array(p_res["confusion_matrix"]), "Model B: Pretrained DistilBERT", "Oranges"),
]
for ax, (cm, title, cmap) in zip(axes, configs):
    im = ax.imshow(cm, cmap=cmap)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(
        xticks=np.arange(3),
        yticks=np.arange(3),
        xticklabels=LABELS,
        yticklabels=LABELS,
        xlabel="Predicted",
        ylabel="True",
        title=title,
    )
    thresh = cm.max() / 2.0
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:d}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Qualitative Sanity Check on Custom Examples ---
sample_inputs = [
    "Just finished watching the new sci-fi movie and honestly it blew my mind! Best soundtrack of the year.",
    "Waited two hours for my delivery and the food arrived completely cold. Never ordering from here again.",
    "The quarterly earnings report will be published next Tuesday at 9 AM Eastern Time.",
    "Oh great, another software update that broke my Wi-Fi right before an important meeting. Just what I needed!",
]

scratch_model.eval()
pretrained_model.eval()

rows = []
for text in sample_inputs:
    with torch.no_grad():
        x_a = torch.tensor([encode_text(text, vocab, MAX_LEN)], dtype=torch.long, device=device)
        probs_a = torch.softmax(scratch_model(x_a), dim=-1).squeeze(0).cpu().numpy()

        enc_b = pretrained_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
        enc_b = {k: v.to(device) for k, v in enc_b.items()}
        probs_b = torch.softmax(pretrained_model(**enc_b).logits, dim=-1).squeeze(0).cpu().numpy()

    rows.append({
        "Input Text": text[:70] + ("..." if len(text) > 70 else ""),
        "Scratch Pred": f"{LABELS[int(probs_a.argmax())]} ({probs_a.max()*100:.1f}%)",
        "Pretrained Pred": f"{LABELS[int(probs_b.argmax())]} ({probs_b.max()*100:.1f}%)",
    })

display(pd.DataFrame(rows))

## 5. Analysis: Scratch Transformer vs. Pretrained DistilBERT

### Why Does the Pretrained Model Outperform the Scratch Model?
1. **Rich Prior Language Knowledge:** `distilbert-base-uncased` was pretrained via Masked Language Modeling on massive text corpora (Wikipedia + BookCorpus). Before seeing a single labeled tweet, it already understands grammar, negation (`"not bad"`), context-dependent words, and sarcasm cues.
2. **Subword Tokenization (WordPiece):** Model A uses a fixed word vocabulary built from 6,000 tweets, so rare words, slang, and spelling variations become `<UNK>`. DistilBERT splits unknown words into meaningful subwords.
3. **Data Efficiency of Transformers:** Self-attention layers have high capacity and minimal inductive bias, making them data-hungry when trained from random initialization. With only 6,000 training tweets, Model A learns surface keyword correlations well but struggles on nuanced or neutral vs. weakly polar tweets.

### Efficiency vs. Accuracy Trade-Off
- **Model Size & Speed:** Model A has ~40x–60x fewer parameters than DistilBERT (~1M–1.5M vs. ~66.9M), trains very fast even on a CPU, and achieves lower single-sample inference latency.
- **Practical Takeaway:** Training a Transformer from scratch is an invaluable educational exercise to understand scaled dot-product attention, masking, and residual blocks — and can be useful for ultra-low-latency or domain-specific non-text sequences. For real-world NLP with limited labeled data, however, fine-tuning a compact pretrained Transformer provides a decisive leap in accuracy and Macro-F1.

---
### Next Step: Launch the Interactive Streamlit Dashboard
Once all cells above have finished running and saved `scratch_model.pt`, `vocab.json`, `pretrained_model/`, and `results.json`, run:
```bash
streamlit run app.py
```